In [1]:
print("Ok")

Ok


In [2]:
%pwd

'c:\\Python_program\\Project2\\Source-Code-Analysis-Bot-\\testing'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'c:\\Python_program\\Project2\\Source-Code-Analysis-Bot-'

In [6]:
from git import Repo
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI

In [8]:
from langchain_classic.memory import ConversationSummaryMemory

In [10]:
from langchain_classic.chains import ConversationalRetrievalChain

In [11]:
os.chdir("testing")

In [12]:
%pwd

'c:\\Python_program\\Project2\\Source-Code-Analysis-Bot-\\testing'

In [13]:
%mkdir test_repo

In [14]:
repo_path="test_repo/"
repo=Repo.clone_from("https://github.com/mdasmar/AI-Interview-Coach.git",to_path=repo_path)

In [15]:
loader=GenericLoader.from_filesystem(repo_path,
glob="**/*",
suffixes=[".py"],
parser=LanguageParser(language=Language.PYTHON,parser_threshold=500))

In [16]:
docs=loader.load()

In [17]:
docs

[Document(metadata={'source': 'test_repo\\analytics.py', 'language': <Language.PYTHON: 'python'>}, page_content='def generate_statistics(history):\n\n    total_questions = len(history)\n\n    if total_questions == 0:\n        return {\n            "total": 0,\n            "correct": 0,\n            "percentage": 0,\n            "easy": (0, 0),\n            "medium": (0, 0),\n            "hard": (0, 0),\n            "topics": {},\n            "strong_topics": [],\n            "weak_topics": []\n        }\n\n    correct_answers = 0\n\n    easy_total = 0\n    easy_correct = 0\n\n    medium_total = 0\n    medium_correct = 0\n\n    hard_total = 0\n    hard_correct = 0\n\n    topics = {}\n\n    for item in history:\n\n        topic = item["topic"]\n        difficulty = item["difficulty"]\n        is_correct = item["is_correct"]\n\n        if is_correct:\n            correct_answers += 1\n\n        if difficulty == "Easy":\n\n            easy_total += 1\n\n            if is_correct:\n        

In [18]:
docs_splitter=RecursiveCharacterTextSplitter.from_language(language=Language.PYTHON,
chunk_size=500,
chunk_overlap=20)

In [19]:
texts=docs_splitter.split_documents(docs)

In [20]:
texts

[Document(metadata={'source': 'test_repo\\analytics.py', 'language': <Language.PYTHON: 'python'>}, page_content='def generate_statistics(history):\n\n    total_questions = len(history)\n\n    if total_questions == 0:\n        return {\n            "total": 0,\n            "correct": 0,\n            "percentage": 0,\n            "easy": (0, 0),\n            "medium": (0, 0),\n            "hard": (0, 0),\n            "topics": {},\n            "strong_topics": [],\n            "weak_topics": []\n        }\n\n    correct_answers = 0\n\n    easy_total = 0\n    easy_correct = 0\n\n    medium_total = 0\n    medium_correct = 0'),
 Document(metadata={'source': 'test_repo\\analytics.py', 'language': <Language.PYTHON: 'python'>}, page_content='hard_total = 0\n    hard_correct = 0\n\n    topics = {}\n\n    for item in history:\n\n        topic = item["topic"]\n        difficulty = item["difficulty"]\n        is_correct = item["is_correct"]\n\n        if is_correct:\n            correct_answers +=

In [21]:
len(texts)

156

In [22]:
embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\KIIT\AppData\Local\Temp\ipykernel_10680\3123569051.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
c:\Users\KIIT\anaconda3\envs\codeAnalyse\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3193.48it/s]


In [23]:
vecstore=Chroma.from_documents(texts,embedding=embeddings,persist_directory="DB")

In [33]:
from dotenv import load_dotenv
load_dotenv()

True

In [35]:
GOOGLE_API_KEY=os.environ.get("GOOGLE_API_KEY")

In [36]:
os.environ["GOOGLE_API_KEY"]=GOOGLE_API_KEY

In [37]:
llm=ChatGoogleGenerativeAI(model="gemini-3.5-flash")

In [38]:
memory=ConversationSummaryMemory(llm=llm,memory_key="chat_history",return_messages=True)

C:\Users\KIIT\AppData\Local\Temp\ipykernel_10680\1926954105.py:1: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory=ConversationSummaryMemory(llm=llm,memory_key="chat_history",return_messages=True)


In [40]:
qa=ConversationalRetrievalChain.from_llm(llm,retriever=vecstore.as_retriever(search_type="mmr",search_kwargs={"k":8}),memory=memory)

In [41]:
question="what is extract_candidate_name function?"

In [42]:
result=qa(question)
print(result["answer"])

C:\Users\KIIT\AppData\Local\Temp\ipykernel_10680\612839885.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result=qa(question)


Based on the provided code, `extract_candidate_name(resume_text)` is a function designed to parse a candidate's name from the text of a resume.

### How it works:
1. **Defines Common Headings:** It maintains a set of common resume section headings (such as `"resume"`, `"profile"`, `"education"`, `"experience"`, etc.) to ignore or filter out.
2. **Inspects Top Lines:** It iterates through the first 15 lines of the provided `resume_text`, stripping whitespace and skipping blank lines.
3. **Validates Name Format:** It uses a regular expression (`re.fullmatch(r"[A-Za-z][A-Za-z .'-]{1,60}", line)`) to check if a line matches valid name criteria (e.g., standard alphabetical characters, spaces, dots, apostrophes, or hyphens).
4. **Returns the Name or Default:** 
   * If a matching line is found, it cleans up extra spaces and returns the extracted name (e.g., returning `"Asmar"` from the demo resume).
   * If no valid name is found within the first 15 lines, it defaults to returning the string